# ClickHouse Columnar Storage — Code Analysis

## Overview

This project analyzes the internal storage mechanism of ClickHouse by examining its write path and core storage components in the MergeTree engine.

---

## 1. writeTempPartImpl()

This function is responsible for creating a temporary data part during insertion.

### Key Responsibilities:
- Receives data blocks from input
- Sorts data based on ORDER BY key
- Splits data into granules
- Writes column-wise data to disk
- Generates metadata (min-max index, marks)

### Key Insight:
This function defines how data is physically organized on disk, directly impacting query performance.

---

## 2. Sorting and Indexing

Before writing to disk:
- Data is sorted based on `(event_date, user_id)`
- Min-max metadata is created for pruning

### Impact:
Sorting enables efficient filtering by allowing the system to skip irrelevant data blocks.

---

## 3. Columnar Storage (MergeTreeDataPartWriterWide)

### Behavior:
- Each column is stored separately
- Files created:
  - `.bin` → column data
  - `.mrk` → index marks

### Impact:
- Only required columns are read
- Reduces disk I/O

---

## 4. Granules and Sparse Index

### Concept:
- Data is divided into granules (default 8192 rows)
- Each granule has one index entry (mark)

### Impact:
- Enables skipping of entire data blocks
- Improves query efficiency

### Tradeoff:
- Smaller granules → better precision, more metadata
- Larger granules → less metadata, more scanning

---

## 5. Compression

Each column is compressed independently.

### Impact:
- Improves storage efficiency
- Reduces disk read size (read_bytes)

### Tradeoff:
- Adds CPU cost during decompression

---

## 6. Wide vs Compact Parts

### Wide Parts:
- Columns stored separately
- Better compression

### Compact Parts:
- Multiple columns stored together
- Fewer files

### Tradeoff:
- Wide → efficient reads
- Compact → efficient inserts

---

## 7. End-to-End Write Path

1. Insert query received  
2. Data converted into blocks  
3. Sorting applied  
4. Data split into granules  
5. Columns written to disk  
6. Metadata generated (marks, min-max)  
7. Part stored on disk  

---

## Final Insight

ClickHouse performance is driven by minimizing disk I/O through:
- Columnar storage
- Sorting
- Sparse indexing
- Compression

> The system optimizes performance by reducing how much data is read, rather than increasing computation speed.